# Car Model Recognition — Data Exploration
BIL462 | Group: Junkers

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from pathlib import Path

import torch
from torchvision.utils import make_grid

from src.dataset import StanfordCarsDataset, get_transforms, build_dataloaders

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded')

In [ ]:
dataset = StanfordCarsDataset(
    root=cfg['data']['dataset_dir'],
    split='train',
    transform=get_transforms(cfg['data']['image_size'], 'val', {})
)

print(f'Total samples: {len(dataset)}')
print(f'Number of classes: {len(dataset.classes)}')
print(f'Sample classes: {dataset.classes[:5]}')

In [ ]:
# Class distribution
labels = [label for _, label in dataset.samples]
counts = Counter(labels)
count_vals = [counts[i] for i in range(len(dataset.classes))]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(range(len(count_vals)), sorted(count_vals))
axes[0].set_title('Class Distribution (sorted)')
axes[0].set_xlabel('Class rank')
axes[0].set_ylabel('Sample count')

axes[1].hist(count_vals, bins=30)
axes[1].set_title('Distribution of samples per class')
axes[1].set_xlabel('Samples per class')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../logs/class_distribution.png', dpi=120)
plt.show()

print(f'Min samples: {min(count_vals)}, Max: {max(count_vals)}, Mean: {np.mean(count_vals):.1f}')

In [ ]:
# Sample grid visualization
indices = np.random.choice(len(dataset), 16, replace=False)
images = torch.stack([dataset[i][0] for i in indices])
class_names_shown = [dataset.classes[dataset[i][1]] for i in indices]

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
images_denorm = (images * std + mean).clamp(0, 1)

grid = make_grid(images_denorm, nrow=4, padding=4)
fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(grid.permute(1, 2, 0))
ax.axis('off')
ax.set_title('Random Sample Grid')

for i, name in enumerate(class_names_shown):
    row, col = divmod(i, 4)
    short = ' '.join(name.split()[-3:])  # last 3 words
    ax.text((col * (224 + 4)) + 4, (row * (224 + 4)) + 215,
            short, color='white', fontsize=7,
            bbox=dict(facecolor='black', alpha=0.6, pad=1))

plt.tight_layout()
plt.savefig('../logs/sample_grid.png', dpi=120)
plt.show()

In [ ]:
# Augmentation preview
aug_transform = get_transforms(cfg['data']['image_size'], 'train', cfg['augmentation'])
val_transform = get_transforms(cfg['data']['image_size'], 'val', {})

from PIL import Image
sample_path, sample_label = dataset.samples[0]
orig_img = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle(f'Augmentation Samples: {dataset.classes[sample_label]}')

val_tensor = val_transform(orig_img)
val_img = (val_tensor * std + mean).clamp(0, 1).permute(1, 2, 0)
axes[0, 0].imshow(val_img)
axes[0, 0].set_title('Original (val)')
axes[0, 0].axis('off')

for i in range(1, 10):
    row, col = divmod(i, 5)
    aug_tensor = aug_transform(orig_img)
    aug_img = (aug_tensor * std + mean).clamp(0, 1).permute(1, 2, 0)
    axes[row, col].imshow(aug_img)
    axes[row, col].set_title(f'Aug {i}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('../logs/augmentation_preview.png', dpi=120)
plt.show()